# Direct Weights Workflow

Use this when a notebook is generating portfolio weights directly instead of generating forecasts first.

In [ ]:
from pathlib import Path
import pandas as pd

from portfolio_toolkit import PortfolioWeights, backtest_weights, baseline_weights, start_run, validate_weights_frame, write_backtest_artifacts, log_backtest

repo_root = Path('../../').resolve()
dataset_name = 'shared_set_1'
output_dir = repo_root / 'runs' / 'direct_weights_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

# Replace this with your own notebook-generated weights.
weights = baseline_weights(dataset_name, 'equal_weight', repo_root=repo_root)
validated_weights = validate_weights_frame(weights.weights, dataset_name=dataset_name, repo_root=repo_root)
portfolio = PortfolioWeights(validated_weights, dataset_name=dataset_name, strategy_name='direct_weights_model')

In [ ]:
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
artifact_paths = write_backtest_artifacts(result, output_dir)

with start_run('direct_weights_model', dataset_name, tags={'workflow': 'direct_weights'}, repo_root=repo_root):
    import mlflow
    mlflow.log_params({'split': 'test', 'cost_bps': 10.0, 'source': 'direct_weights'})
    log_backtest(result)

result.metrics

In [ ]:
# Validation cell
assert Path(artifact_paths['quantstats_report']).exists()
assert abs(result.weights.sum(axis=1).iloc[0] - 1.0) < 1e-9
print('Direct weights workflow validated.')